In [1]:
# I'm starting a fresh notebook for cleaning, so I need to reload everything again
from datasets import load_dataset
import pandas as pd

# I'm downloading (or reusing the cached copy of) the ToN_IoT dataset
dataset = load_dataset("codymlewis/TON_IoT_network")

# I'm converting the "train" split into a pandas DataFrame so I can clean it
df = dataset["train"].to_pandas()

# I'm checking the shape to confirm I still have all 211043 rows and 44 columns
df.shape

C:\Users\Admin\anaconda3\envs\fl-ids\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(211043, 44)

In [3]:
# I'm dropping these columns because:
# - src_ip / dst_ip are raw IP addresses — if I keep them, my model might just
#   memorize specific IPs instead of actually learning attack patterns
# - dns_query, ssl_subject, ssl_issuer, http_uri, http_user_agent are messy,
#   free-text fields that are too specific/sparse to use as simple numeric
#   features right now, so I'm skipping them for this version
# - type is the detailed attack category, but I've decided to frame this as
#   binary classification (normal vs. attack) using the 'label' column instead
drop_cols = ['src_ip', 'dst_ip', 'dns_query', 'ssl_subject', 'ssl_issuer',
             'http_uri', 'http_user_agent', 'type']

# I'm actually removing those columns from my working copy now
df = df.drop(columns=drop_cols)

# I'm confirming the new shape — same rows, fewer columns
df.shape

(211043, 36)

In [5]:
# checking for missing values per column
df.isnull().sum().sort_values(ascending=False).head(15)

src_port        0
dst_port        0
proto           0
service         0
duration        0
src_bytes       0
dst_bytes       0
conn_state      0
missed_bytes    0
src_pkts        0
src_ip_bytes    0
dst_pkts        0
dst_ip_bytes    0
dns_qclass      0
dns_qtype       0
dtype: int64

In [7]:
# checking for "-" placeholder values in text columns
for col in df.select_dtypes(include='object').columns:
    count = (df[col] == '-').sum()
    if count > 0:
        print(col, count)

service 132032
dns_AA 176030
dns_RD 176030
dns_RA 176030
dns_rejected 176030
ssl_version 210642
ssl_cipher 210642
ssl_resumed 210642
ssl_established 210642
http_trans_depth 210740
http_method 210756
http_version 210745
http_orig_mime_types 211027
http_resp_mime_types 210839
weird_name 210687
weird_addl 210886
weird_notice 210687


In [9]:
# these columns are 99.8%+ empty, not useful — dropping
mostly_empty = ['ssl_version', 'ssl_cipher', 'ssl_resumed', 'ssl_established',
                 'http_trans_depth', 'http_method', 'http_version',
                 'http_orig_mime_types', 'http_resp_mime_types',
                 'weird_name', 'weird_addl', 'weird_notice']

df = df.drop(columns=mostly_empty)
df.shape

(211043, 24)

In [13]:
# for the remaining "-" columns, I'm treating "-" as its own category
# instead of dropping them, since "no DNS/service used" can be meaningful
keep_with_dash = ['service', 'dns_AA', 'dns_RD', 'dns_RA', 'dns_rejected']

for col in keep_with_dash:
    df[col] = df[col].replace('-', 'none')

In [15]:
# quick look at remaining columns and their types
df.dtypes

src_port                    int64
dst_port                    int64
proto                      object
service                    object
duration                  float64
src_bytes                   int64
dst_bytes                   int64
conn_state                 object
missed_bytes                int64
src_pkts                    int64
src_ip_bytes                int64
dst_pkts                    int64
dst_ip_bytes                int64
dns_qclass                  int64
dns_qtype                   int64
dns_rcode                   int64
dns_AA                     object
dns_RD                     object
dns_RA                     object
dns_rejected               object
http_request_body_len       int64
http_response_body_len      int64
http_status_code            int64
label                       int64
dtype: object

In [17]:
# checking how many unique values each text column has —
# this decides how I encode them
text_cols = ['proto', 'service', 'conn_state', 'dns_AA', 'dns_RD', 'dns_RA', 'dns_rejected']
for col in text_cols:
    print(col, df[col].nunique())
    

proto 3
service 9
conn_state 13
dns_AA 3
dns_RD 3
dns_RA 3
dns_rejected 3


In [19]:
# converting text columns into 0/1 numeric columns
df = pd.get_dummies(df, columns=text_cols)
df.shape

(211043, 54)

In [21]:
# splitting features (X) and the label (y)
X = df.drop(columns=['label'])
y = df['label']

X.shape, y.shape

((211043, 53), (211043,))

In [23]:
# scaling numeric features so they're on a similar scale — helps training
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled.shape

(211043, 53)

In [25]:
# splitting into train and test sets
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

X_train.shape, X_test.shape

((168834, 53), (42209, 53))

In [29]:
# saving the cleaned data so I don't have to redo all this next time
import numpy as np

np.save('X_train.npy', X_train)
np.save('X_test.npy', X_test)
np.save('y_train.npy', y_train.to_numpy())
np.save('y_test.npy', y_test.to_numpy())